In [1]:
import sqlite3
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore")

# Database path
DB_PATH = r"C:\Users\ADMIN\smart agricultural analytics\agriculture.db"

# Output folders
OUTPUT_DIR = "leakage_audit_outputs"
MODEL_DIR = "stage26_models"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Setup completed.")

Setup completed.


In [2]:
conn = sqlite3.connect(DB_PATH)

crop_df = pd.read_sql_query(
    "SELECT * FROM crop_recommendation_selected",
    conn
)

yield_df = pd.read_sql_query(
    "SELECT * FROM agri_yield_train_selected",
    conn
)

yield_test_df = pd.read_sql_query(
    "SELECT * FROM agri_yield_test_selected",
    conn
)

conn.close()

print("Crop dataset:", crop_df.shape)
print("Yield training dataset:", yield_df.shape)
print("Yield test dataset:", yield_test_df.shape)

display(crop_df.head())
display(yield_df.head())

Crop dataset: (2200, 13)
Yield training dataset: (10000, 11)
Yield test dataset: (2000, 10)


,humidity,K,temperature,N,P_K_Ratio,NPK_Total,Temperature_Rainfall,rainfall,N_K_Ratio,Humidity_Rainfall,Temperature_Humidity,P,label
0,82.002744,43,20.879744,90,0.976744,175,4237.241986,202.935536,2.093023,16641.270870,1712.196283,42,rice
1,80.319644,41,21.770462,85,1.414634,184,4934.395694,226.655537,2.073171,18204.892093,1748.595734,58,rice
2,82.320763,44,23.004459,60,1.250000,159,6072.354751,263.964248,1.363636,21729.738241,1893.744627,55,rice
3,80.158363,40,26.491096,74,0.875000,149,6433.734530,242.864034,1.850000,19467.583326,2123.482908,35,rice
4,81.604873,42,20.130175,78,1.000000,162,5288.545993,262.717340,1.857143,21439.015172,1642.720357,42,rice


,NDVI_Temperature,OrganicMatter_Sand,Temperature_Rainfall,sand_pct,NDVI_Rainfall,soil_ph,organic_matter,Humidity_Rainfall,Temperature_Humidity,rainfall,yield
0,11.592,110.656,2777.04,41.6,50.692,6.25,2.66,7714.0,1764.0,110.2,4105.8
1,15.871,147.132,3397.47,40.2,74.517,6.70,3.66,9851.4,2098.2,126.3,4247.8
2,17.664,138.000,3019.44,40.0,70.016,7.21,3.45,7220.4,1821.6,109.4,4021.8
3,18.352,141.112,4596.88,56.9,96.286,5.73,2.48,12268.7,2338.4,155.3,4326.8
4,15.500,110.466,4315.00,30.6,107.012,6.88,3.61,13808.0,2000.0,172.6,3708.0


In [3]:
print("CROP DATASET")
print("Missing values:")
print(crop_df.isnull().sum())

print("\nDuplicate rows:", crop_df.duplicated().sum())

print("\nYIELD TRAINING DATASET")
print("Missing values:")
print(yield_df.isnull().sum())

print("\nDuplicate rows:", yield_df.duplicated().sum())

print("\nYIELD TEST DATASET")
print("Missing values:")
print(yield_test_df.isnull().sum())

print("\nDuplicate rows:", yield_test_df.duplicated().sum())

CROP DATASET
Missing values:
humidity                0
K                       0
temperature             0
N                       0
P_K_Ratio               0
NPK_Total               0
Temperature_Rainfall    0
rainfall                0
N_K_Ratio               0
Humidity_Rainfall       0
Temperature_Humidity    0
P                       0
label                   0
dtype: int64

Duplicate rows: 0

YIELD TRAINING DATASET
Missing values:
NDVI_Temperature        0
OrganicMatter_Sand      0
Temperature_Rainfall    0
sand_pct                0
NDVI_Rainfall           0
soil_ph                 0
organic_matter          0
Humidity_Rainfall       0
Temperature_Humidity    0
rainfall                0
yield                   0
dtype: int64

Duplicate rows: 0

YIELD TEST DATASET
Missing values:
NDVI_Temperature        0
OrganicMatter_Sand      0
Temperature_Rainfall    0
sand_pct                0
NDVI_Rainfall           0
soil_ph                 0
organic_matter          0
Humidity_Rainfall       0

In [4]:
# Crop classification
X_crop = crop_df.drop(columns=["label"])
y_crop = crop_df["label"]

# Yield regression
X_yield = yield_df.drop(columns=["yield"])
y_yield = yield_df["yield"]

# Check feature columns
print("Crop features:")
print(X_crop.columns.tolist())

print("\nYield features:")
print(X_yield.columns.tolist())

print("\nCrop target classes:", y_crop.nunique())

Crop features:
['humidity', 'K', 'temperature', 'N', 'P_K_Ratio', 'NPK_Total', 'Temperature_Rainfall', 'rainfall', 'N_K_Ratio', 'Humidity_Rainfall', 'Temperature_Humidity', 'P']

Yield features:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall']

Crop target classes: 22


In [5]:
# Check for non-numeric input columns
print("Crop non-numeric columns:")
print(X_crop.select_dtypes(exclude=np.number).columns.tolist())

print("\nYield non-numeric columns:")
print(X_yield.select_dtypes(exclude=np.number).columns.tolist())

# Check target missing values
print("\nMissing crop labels:", y_crop.isnull().sum())
print("Missing yield values:", y_yield.isnull().sum())

# Check infinite values
print("\nCrop infinite values:", np.isinf(X_crop.select_dtypes(include=np.number)).sum().sum())
print("Yield infinite values:", np.isinf(X_yield.select_dtypes(include=np.number)).sum().sum())

Crop non-numeric columns:
[]

Yield non-numeric columns:
[]

Missing crop labels: 0
Missing yield values: 0

Crop infinite values: 0
Yield infinite values: 0


In [6]:
X_crop_train, X_crop_test, y_crop_train, y_crop_test = train_test_split(
    X_crop,
    y_crop,
    test_size=0.20,
    random_state=42,
    stratify=y_crop
)

print("Crop training rows:", X_crop_train.shape[0])
print("Crop testing rows:", X_crop_test.shape[0])

print("\nTraining class distribution:")
print(y_crop_train.value_counts().sort_index())

print("\nTesting class distribution:")
print(y_crop_test.value_counts().sort_index())

Crop training rows: 1760
Crop testing rows: 440

Training class distribution:
label
apple          80
banana         80
blackgram      80
chickpea       80
coconut        80
coffee         80
cotton         80
grapes         80
jute           80
kidneybeans    80
lentil         80
maize          80
mango          80
mothbeans      80
mungbean       80
muskmelon      80
orange         80
papaya         80
pigeonpeas     80
pomegranate    80
rice           80
watermelon     80
Name: count, dtype: int64

Testing class distribution:
label
apple          20
banana         20
blackgram      20
chickpea       20
coconut        20
coffee         20
cotton         20
grapes         20
jute           20
kidneybeans    20
lentil         20
maize          20
mango          20
mothbeans      20
mungbean       20
muskmelon      20
orange         20
papaya         20
pigeonpeas     20
pomegranate    20
rice           20
watermelon     20
Name: count, dtype: int64


In [7]:
X_yield_train, X_yield_test, y_yield_train, y_yield_test = train_test_split(
    X_yield,
    y_yield,
    test_size=0.20,
    random_state=42
)

print("Yield training rows:", X_yield_train.shape[0])
print("Yield testing rows:", X_yield_test.shape[0])

Yield training rows: 8000
Yield testing rows: 2000


In [8]:
def check_overlap(X_train, X_test, name):

    train_rows = set(
        pd.util.hash_pandas_object(X_train, index=False)
    )

    test_rows = set(
        pd.util.hash_pandas_object(X_test, index=False)
    )

    overlap = train_rows.intersection(test_rows)

    print(name)
    print("Identical feature rows in both splits:", len(overlap))

    if len(overlap) > 0:
        print("WARNING: Inspect these duplicate observations.")
    else:
        print("No identical feature rows found across splits.")

    print()


check_overlap(X_crop_train, X_crop_test, "Crop dataset")
check_overlap(X_yield_train, X_yield_test, "Yield dataset")

Crop dataset
Identical feature rows in both splits: 0
No identical feature rows found across splits.

Yield dataset
Identical feature rows in both splits: 0
No identical feature rows found across splits.



In [9]:
def create_pipeline(model):

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    return pipeline

In [10]:
crop_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

crop_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

crop_results = []

for name, model in crop_models.items():

    pipeline = create_pipeline(model)

    scores = cross_validate(
        pipeline,
        X_crop_train,
        y_crop_train,
        cv=crop_cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision_weighted",
            "recall": "recall_weighted",
            "f1": "f1_weighted"
        },
        n_jobs=-1
    )

    crop_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean()
    })

crop_results_df = pd.DataFrame(crop_results)

crop_results_df = crop_results_df.sort_values(
    by="Accuracy",
    ascending=False
)

display(crop_results_df)

crop_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "crop_cv_results.csv"),
    index=False
)

best_crop_name = crop_results_df.iloc[0]["Model"]

print("Best crop model based on training CV:", best_crop_name)

,Model,Accuracy,Precision,Recall,F1
2,Random Forest,0.989205,0.990212,0.989205,0.989074
1,Decision Tree,0.978977,0.980454,0.978977,0.979018
3,SVM,0.976136,0.979766,0.976136,0.975790
0,Logistic Regression,0.970455,0.973240,0.970455,0.970316
4,KNN,0.964773,0.969507,0.964773,0.964764


Best crop model based on training CV: Random Forest


In [11]:
best_crop_model = crop_models[best_crop_name]

crop_pipeline = create_pipeline(best_crop_model)

crop_pipeline.fit(X_crop_train, y_crop_train)

crop_predictions = crop_pipeline.predict(X_crop_test)

print("CROP CLASSIFICATION HOLDOUT RESULTS")

print("Accuracy:",
      accuracy_score(y_crop_test, crop_predictions))

print("Weighted Precision:",
      precision_score(y_crop_test, crop_predictions, average="weighted"))

print("Weighted Recall:",
      recall_score(y_crop_test, crop_predictions, average="weighted"))

print("Weighted F1:",
      f1_score(y_crop_test, crop_predictions, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_crop_test, crop_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_crop_test, crop_predictions))

CROP CLASSIFICATION HOLDOUT RESULTS
Accuracy: 0.9886363636363636
Weighted Precision: 0.989742141916055
Weighted Recall: 0.9886363636363636
Weighted F1: 0.988661267581768

Classification Report:
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.90      0.95        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      0.90      0.95        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.87      1.00      0.93      

In [12]:
crop_predictions_df = pd.DataFrame({
    "Actual": y_crop_test.values,
    "Predicted": crop_predictions
})

crop_predictions_df.to_csv(
    os.path.join(OUTPUT_DIR, "crop_holdout_predictions.csv"),
    index=False
)

print("Crop predictions saved.")

Crop predictions saved.


In [13]:
yield_models = {
    "Dummy Regressor": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.03,
        random_state=42
    )
}

yield_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

yield_results = []

for name, model in yield_models.items():

    pipeline = create_pipeline(model)

    scores = cross_validate(
        pipeline,
        X_yield_train,
        y_yield_train,
        cv=yield_cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1
    )

    yield_results.append({
        "Model": name,
        "MAE": -scores["test_mae"].mean(),
        "RMSE": -scores["test_rmse"].mean(),
        "R2": scores["test_r2"].mean()
    })

yield_results_df = pd.DataFrame(yield_results)

yield_results_df = yield_results_df.sort_values(
    by="R2",
    ascending=False
)

display(yield_results_df)

yield_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "yield_cv_results.csv"),
    index=False
)

best_yield_name = yield_results_df.iloc[0]["Model"]

print("Best yield model based on training CV:", best_yield_name)

,Model,MAE,RMSE,R2
4,Gradient Boosting,378.889242,436.666585,0.000369
0,Dummy Regressor,379.378635,436.795868,-0.000218
1,Linear Regression,379.444813,437.129258,-0.001773
3,Random Forest,385.265449,445.935781,-0.042596
2,Decision Tree,503.506012,618.916023,-1.009135


Best yield model based on training CV: Gradient Boosting


In [14]:
best_yield_model = yield_models[best_yield_name]

yield_pipeline = create_pipeline(best_yield_model)

yield_pipeline.fit(X_yield_train, y_yield_train)

yield_predictions = yield_pipeline.predict(X_yield_test)

mae = mean_absolute_error(y_yield_test, yield_predictions)

rmse = np.sqrt(
    mean_squared_error(y_yield_test, yield_predictions)
)

r2 = r2_score(y_yield_test, yield_predictions)

print("YIELD REGRESSION HOLDOUT RESULTS")

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

YIELD REGRESSION HOLDOUT RESULTS
MAE: 376.9240657383142
RMSE: 435.3874922018455
R2: -0.0004680156162952809


In [15]:
dummy_pipeline = create_pipeline(
    DummyRegressor(strategy="mean")
)

dummy_pipeline.fit(X_yield_train, y_yield_train)

dummy_predictions = dummy_pipeline.predict(X_yield_test)

dummy_mae = mean_absolute_error(
    y_yield_test,
    dummy_predictions
)

dummy_rmse = np.sqrt(
    mean_squared_error(y_yield_test, dummy_predictions)
)

dummy_r2 = r2_score(
    y_yield_test,
    dummy_predictions
)

comparison = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Selected Model": [mae, rmse, r2],
    "Dummy Baseline": [dummy_mae, dummy_rmse, dummy_r2]
})

display(comparison)

comparison.to_csv(
    os.path.join(OUTPUT_DIR, "yield_holdout_comparison.csv"),
    index=False
)

,Metric,Selected Model,Dummy Baseline
0,MAE,376.924066,376.930043
1,RMSE,435.387492,435.325183
2,R2,-0.000468,-0.000182


In [16]:
# Crop model trained on all selected crop records
final_crop_pipeline = create_pipeline(
    crop_models[best_crop_name]
)

final_crop_pipeline.fit(X_crop, y_crop)

joblib.dump(
    final_crop_pipeline,
    os.path.join(MODEL_DIR, "crop_classifier.joblib")
)


# Yield model trained on all selected yield training records
final_yield_pipeline = create_pipeline(
    yield_models[best_yield_name]
)

final_yield_pipeline.fit(X_yield, y_yield)

joblib.dump(
    final_yield_pipeline,
    os.path.join(MODEL_DIR, "yield_predictor.joblib")
)

print("Final model pipelines saved.")

Final model pipelines saved.


In [17]:
external_yield_predictions = final_yield_pipeline.predict(
    yield_test_df[X_yield.columns]
)

yield_test_predictions_df = pd.DataFrame({
    "Predicted_Yield": external_yield_predictions
})

yield_test_predictions_df.to_csv(
    os.path.join(OUTPUT_DIR, "yield_test_predictions.csv"),
    index=False
)

print("Yield test predictions saved.")

Yield test predictions saved.


In [18]:
audit_summary = pd.DataFrame({
    "Audit Item": [
        "Crop training rows",
        "Crop holdout rows",
        "Yield training rows",
        "Yield holdout rows",
        "Crop best model",
        "Yield best model",
        "Crop CV completed",
        "Yield CV completed",
        "Preprocessing inside pipeline",
        "Feature selection inside pipeline"
    ],
    "Result": [
        len(X_crop_train),
        len(X_crop_test),
        len(X_yield_train),
        len(X_yield_test),
        best_crop_name,
        best_yield_name,
        "Yes",
        "Yes",
        "Yes",
        "No - selected tables already supplied"
    ]
})

display(audit_summary)

audit_summary.to_csv(
    os.path.join(OUTPUT_DIR, "leakage_audit_summary.csv"),
    index=False
)

print("\nLeakage audit evaluation completed.")
print("Results folder:", OUTPUT_DIR)
print("Model folder:", MODEL_DIR)

,Audit Item,Result
0,Crop training rows,1760
1,Crop holdout rows,440
2,Yield training rows,8000
3,Yield holdout rows,2000
4,Crop best model,Random Forest
5,Yield best model,Gradient Boosting
6,Crop CV completed,Yes
7,Yield CV completed,Yes
8,Preprocessing inside pipeline,Yes
9,Feature selection inside pipeline,No - selected tables already supplied



Leakage audit evaluation completed.
Results folder: leakage_audit_outputs
Model folder: stage26_models
